In [2]:
! pip install autogluon


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of opentelemetry-sdk to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of opentelemetry-sdk to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of openxlab to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.5/259.5 kB 8.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is stil

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np


In [3]:
from google.colab import files

uploaded = files.upload()

Saving dataset.csv to dataset.csv


In [4]:
import pandas as pd

df = pd.read_csv('dataset.csv')

df.head()

,text,sentiment,text_length,clean_text
0,@looby_loo You are so full of good ideas Why ...,positive,69,you are so full of good ideas why didnt i thin...
1,"@marieez um ok, but i might not get there til ...",negative,91,um ok but i might not get there til like cos m...
2,is off to work for the day.,negative,28,is off to work for the day
3,@Dont_Panic42 ugh. You're so weird. Omg I ju...,negative,103,ugh youre so weird omg i just finished cleanin...
4,@fossiloflife tell him that he needs to fork o...,positive,84,tell him that he needs to fork out to the regu...


In [5]:
df.columns

Index(['text', 'sentiment', 'text_length', 'clean_text'], dtype='object')

In [6]:
from sklearn.model_selection import train_test_split

X = df['clean_text']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training Samples:", len(X_train))
print("Testing Samples:", len(X_test))

Training Samples: 24000
Testing Samples: 6000


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=15000,
    ngram_range=(1,2),
    stop_words='english'
)

X_train_tfidf = tfidf.fit_transform(X_train)

X_test_tfidf = tfidf.transform(X_test)

print(X_train_tfidf.shape)

(24000, 15000)


In [8]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(
    max_iter=2000,
    random_state=42
)

lr_model.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=2000, random_state=42)

In [9]:
comment = ["This phone is amazing and works perfectly"]

comment_vector = tfidf.transform(comment)

prediction = lr_model.predict(comment_vector)

print(prediction)

['positive']


In [10]:
from autogluon.tabular import TabularPredictor
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df[['clean_text', 'sentiment']],
    test_size=0.2,
    random_state=42,
    stratify=df['sentiment']
)

print(train_df.shape)
print(test_df.shape)

(24000, 2)
(6000, 2)


In [11]:
predictor = TabularPredictor(
    label='sentiment',
    eval_metric='accuracy'
).fit(
    train_df,
    time_limit=1800  # 30 minutes
)

No path specified. Models will be saved in: "AutogluonModels/ag-20260602_120616"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Apr 30 18:17:14 UTC 2026
CPU Count:          2
Pytorch Version:    2.9.1+cu128
CUDA Version:       CUDA is not available
Memory Avail:       10.61 GB / 12.67 GB (83.8%)
Disk Space Avail:   75.63 GB / 107.72 GB (70.2%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme'  : New in v1.5: The state-of-the-art for tabular data. Massively better than 'best' on datasets <100000 samples by using new Tabular Foundation Models (TFMs) meta-learned on https://tabarena.

In [12]:
predictor.evaluate(test_df)

{'accuracy': 0.748,
 'balanced_accuracy': np.float64(0.7478826462496662),
 'mcc': np.float64(0.49609283794779796),
 'roc_auc': np.float64(0.8304648594447794),
 'f1': 0.7534246575342466,
 'precision': 0.742526518804243,
 'recall': 0.7646474677259185}

In [13]:
predictions = predictor.predict(test_df)

predictions.head()

,sentiment
6697,positive
28982,positive
21847,positive
18268,positive
18324,negative


In [14]:
sample = pd.DataFrame({
    'clean_text': [
        'this product is amazing',
        'worst experience ever',
        'i love this phone',
        'very disappointed'
    ],
    'text_length': [23, 21, 17, 17]
})

predictor.predict(sample)


,sentiment
0,positive
1,negative
2,positive
3,negative


In [15]:
from autogluon.tabular import TabularPredictor

predictor = TabularPredictor.load(
    "/content/AutogluonModels/ag-20260602_120616"
)

In [21]:
user_comment = input("Enter Instagram comment: ")

input_df = pd.DataFrame({
    'clean_text': [user_comment],
    'text_length': [len(user_comment)]
})

prediction = predictor.predict(input_df)

print("\nPredicted Sentiment:", prediction.iloc[0])

Enter Instagram comment: "This actor abuses so much , i hate him"

Predicted Sentiment: negative


In [22]:
import pandas as pd

user_comment = input("Enter Instagram comment: ")

input_df = pd.DataFrame({
    'clean_text': [user_comment],
    'text_length': [len(user_comment)]
})

prediction = predictor.predict(input_df)
probabilities = predictor.predict_proba(input_df)

print("\nComment:", user_comment)
print("Predicted Sentiment:", prediction.iloc[0])

print("\nConfidence Scores:")
print(probabilities)

Enter Instagram comment: "I don't like it as much it's tested better by lot's of people"

Comment: "I don't like it as much it's tested better by lot's of people"
Predicted Sentiment: positive

Confidence Scores:
   negative  positive
0  0.194765  0.805235


In [38]:
import pandas as pd

while True:
    user_comment = input("\nEnter comment (type 'exit' to quit): ")

    if user_comment.lower() == 'exit':
        break

    input_df = pd.DataFrame({
        'clean_text': [user_comment],
        'text_length': [len(user_comment)]
    })

    prediction = predictor.predict(input_df)
    probabilities = predictor.predict_proba(input_df)

    print("Predicted Sentiment:", prediction.iloc[0])
    print(probabilities)


Enter comment (type 'exit' to quit): The movie had great acting , but story was boring
Predicted Sentiment: negative
   negative  positive
0  0.670414  0.329586

Enter comment (type 'exit' to quit): exit


In [24]:
import shutil

shutil.make_archive(
    "autogluon_model",
    "zip",
    "/content/AutogluonModels/ag-20260602_120616"
)

'/content/autogluon_model.zip'

In [26]:
! pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 32.1 MB/s eta 0:00:00


In [28]:
from google.colab import files

files.download("autogluon_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [27]:
import streamlit as st
import pandas as pd
from autogluon.tabular import TabularPredictor

# Load model
@st.cache_resource
def load_model():
    return TabularPredictor.load("predictor")

predictor = load_model()

st.title("📊 Instagram Sentiment Analyzer")

st.write("Analyze Instagram comments using AutoGluon AI")

comment = st.text_area(
    "Enter Instagram Comment",
    height=150
)

if st.button("Analyze Sentiment"):

    if comment.strip():

        input_df = pd.DataFrame({
            "clean_text": [comment],
            "text_length": [len(comment)]
        })

        prediction = predictor.predict(input_df)

        probabilities = predictor.predict_proba(input_df)

        st.success(
            f"Predicted Sentiment: {prediction.iloc[0]}"
        )

        st.subheader("Confidence Scores")

        st.dataframe(probabilities)

        st.bar_chart(probabilities.T)

    else:
        st.warning("Please enter a comment.")

2026-06-02 12:47:31.312 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
This means that the predictor was fit in an AutoGluon version `<=0.3.1`.


FileNotFoundError: [Errno 2] No such file or directory: '/content/predictor/predictor.pkl'